In [ ]:
!pip install -q torch huggingface_hub matplotlib

In [ ]:
import os
from getpass import getpass

if not os.environ.get('HF_TOKEN'):
    token = getpass('HF_TOKEN: ').strip()
    if not token:
        raise RuntimeError('HF_TOKEN is required to download data and upload v16 artifacts')
    os.environ['HF_TOKEN'] = token

print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import os

# Override any of these before running cell 5.
defaults = {
    'CANDIDATES_CSV': '',
    'V16_EPOCHS': '280',
    'V16_BATCH_SIZE': '4096',
    'V16_LR': '0.00045',
    'V16_WEIGHT_DECAY': '0.00030',
    'V16_PAIR_LOSS_WEIGHT': '1.05',
    'V16_MAX_PAIRS_PER_TURN': '12',
    'V16_ENSEMBLE_SIZE': '8',
    'V16_PATIENCE': '36',
    'V16_DROPOUT': '0.13',
    'V16_SCORE_SCALE': '205.0',
    'V16_SEED': '1013',
    'V16_CHECKPOINT_EVERY': '40',
}
for k, v in defaults.items():
    os.environ.setdefault(k, v)

print('Config:', {k: os.environ[k] for k in defaults})

In [ ]:
from huggingface_hub import hf_hub_download
import os

PINNED_DATASET = "data/20260517_074915/candidates_v7.csv"
print(f"Fetching pinned dataset: {PINNED_DATASET}")

csv_path = hf_hub_download(
    repo_id="devaanshpa/orbit-wars-agent",
    repo_type="model",
    filename=PINNED_DATASET,
    token=os.environ["HF_TOKEN"],
)
os.environ['CANDIDATES_CSV'] = csv_path
print(f"Dataset ready: {csv_path}")

In [ ]:
# Read the trainer code from train_v16_ranker.py embedded below.
# This is a self-contained copy so the notebook runs on Kaggle without cloning the repo.
import urllib.request, tempfile, os
trainer_url = 'https://huggingface.co/devaanshpa/orbit-wars-agent/resolve/main/v16/train_v16_ranker.py'
try:
    tmp = tempfile.NamedTemporaryFile(suffix='.py', delete=False)
    urllib.request.urlretrieve(trainer_url, tmp.name)
    with open(tmp.name) as f:
        V16_RANKER_CODE = f.read()
    print(f'Downloaded trainer from HF: {len(V16_RANKER_CODE)} chars')
except Exception:
    # Fallback: read from local file if available
    with open('notebooks/v16/train_v16_ranker.py') as f:
        V16_RANKER_CODE = f.read()
    print(f'Loaded trainer from local: {len(V16_RANKER_CODE)} chars')

In [ ]:
import argparse, os

args = argparse.Namespace(
    csv=os.environ.get('CANDIDATES_CSV', ''),
    prefer_local_data=False,
    export_dir='notebooks/v16/exports',
    epochs=int(os.environ['V16_EPOCHS']),
    batch_size=int(os.environ['V16_BATCH_SIZE']),
    lr=float(os.environ['V16_LR']),
    weight_decay=float(os.environ['V16_WEIGHT_DECAY']),
    pair_loss_weight=float(os.environ['V16_PAIR_LOSS_WEIGHT']),
    max_pairs_per_turn=int(os.environ['V16_MAX_PAIRS_PER_TURN']),
    ensemble_size=int(os.environ['V16_ENSEMBLE_SIZE']),
    patience=int(os.environ['V16_PATIENCE']),
    dropout=float(os.environ['V16_DROPOUT']),
    score_scale=float(os.environ['V16_SCORE_SCALE']),
    seed=int(os.environ['V16_SEED']),
    checkpoint_every=int(os.environ['V16_CHECKPOINT_EVERY']),
    upload=True,
    hf_repo_id='devaanshpa/orbit-wars-agent',
    hf_repo_type='model',
)
print({
    'epochs': args.epochs,
    'ensemble_size': args.ensemble_size,
    'batch_size': args.batch_size,
    'checkpoint_every': args.checkpoint_every,
    'csv': args.csv or '(auto)',
})

In [ ]:
v16_ns = {}
exec(V16_RANKER_CODE, v16_ns)
v16_ns['train'](args)